In [ ]:
import pandas as pd
import numpy as np
import re
import textstat
from decimal import Decimal, ROUND_HALF_UP
from tqdm import tqdm
from datasets import load_dataset
from sentence_transformers import SentenceTransformer, util
from sklearn.model_selection import train_test_split

import spacy
from spacy.lang.en import English

# Tải công cụ NLP
try:
    nlp = spacy.load("en_core_web_sm", disable=["ner", "parser"])
    print("Loaded spaCy model: en_core_web_sm")
except OSError:
    print("Không tìm thấy en_core_web_sm, fallback sang English() + sentencizer")
    nlp = English()
    nlp.add_pipe("sentencizer")

sbert_model = SentenceTransformer("all-MiniLM-L6-v2")

print("--- Đang tải dữ liệu từ Hugging Face ---")
dataset = load_dataset("chillies/IELTS-writing-task-2-evaluation")
df = pd.concat([pd.DataFrame(dataset["train"]), pd.DataFrame(dataset["test"])], ignore_index=True)
print(f"Tổng số mẫu ban đầu: {len(df)}")

In [ ]:
# CELL 2
# 1. Chuẩn hóa văn bản
def basic_clean(text):
    if not isinstance(text, str):
        return ""
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    text = text.replace("\n", " ")
    text = re.sub(r"\s+", " ", text).strip()
    return text

def normalize_eval_text(text):
    if not isinstance(text, str):
        return ""

    text = text.replace("\r\n", "\n").replace("\r", "\n")

    # fix mojibake phổ biến
    text = (
        text.replace("â€™", "'")
            .replace("â€˜", "'")
            .replace("â€œ", '"')
            .replace("â€\x9d", '"')
            .replace("â€“", "-")
            .replace("â€”", "-")
    )

    # markdown / bullet
    text = text.replace("•", "-")
    text = text.replace("###", "")
    text = text.replace("##", "")
    text = text.replace("**", "")
    text = text.replace("__", "")

    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()

df["essay"] = df["essay"].apply(basic_clean)
df["prompt"] = df["prompt"].apply(basic_clean)
df["evaluation"] = df["evaluation"].apply(normalize_eval_text)

# 2. Xóa mẫu lỗi / trùng / quá ngắn
df = df.dropna(subset=["essay", "prompt", "evaluation"]).reset_index(drop=True)
df = df.drop_duplicates(subset=["essay"]).reset_index(drop=True)

df["essay_len"] = df["essay"].apply(lambda x: len(x.split()))
df = df[df["essay_len"] >= 150].reset_index(drop=True)

print(f"Số lượng sau khi làm sạch & lọc trùng: {len(df)}")

Số lượng sau khi làm sạch & lọc trùng: 8609


In [ ]:
# CELL 4
CRITERION_ALIASES = {
    "TR": [
        "task achievement",
        "task response",
        "task response/achievement",
    ],
    "CC": [
        "coherence and cohesion",
        "cohesion and coherence",
    ],
    "LR": [
        "lexical resource",
        "lexical resource (vocabulary)",
        "vocabulary",
    ],
    "GRA": [
        "grammatical range and accuracy",
        "grammar",
        "grammar accuracy",
        "grammatical accuracy",
    ],
}

ALL_ALIASES = []
for aliases in CRITERION_ALIASES.values():
    ALL_ALIASES.extend(aliases)

SECTION_NAME_PATTERN = "|".join(
    sorted((re.escape(x) for x in ALL_ALIASES), key=len, reverse=True)
)

RE_FLAGS = re.IGNORECASE | re.MULTILINE | re.VERBOSE

def clamp_band(x):
    try:
        x = float(x)
    except Exception:
        return np.nan
    if 0.0 <= x <= 9.0:
        return x
    return np.nan

def build_heading_regex(alias):
    return rf"""
^\s*
(?:[-*]\s*)?
{re.escape(alias)}
\s*
(?:\(\s*\d(?:\.\d)?\s*\))?
\s*:
"""

def find_section(text, aliases):
    starts = []
    for alias in aliases:
        pat = build_heading_regex(alias)
        for m in re.finditer(pat, text, flags=RE_FLAGS):
            starts.append((m.start(), m.end(), alias))

    if not starts:
        return None

    starts.sort(key=lambda x: x[0])
    start, end, _ = starts[0]

    next_heading_pat = rf"""
^\s*
(?:[-*]\s*)?
(?:{SECTION_NAME_PATTERN})
\s*
(?:\(\s*\d(?:\.\d)?\s*\))?
\s*:
"""

    m_next = re.search(next_heading_pat, text[end:], flags=RE_FLAGS)
    if m_next:
        section_end = end + m_next.start()
    else:
        section_end = len(text)

    return text[start:section_end].strip()

def extract_score_from_heading(section_text, aliases):
    for alias in aliases:
        patterns = [
            # Task Achievement (7.5):
            rf"^\s*(?:[-*]\s*)?{re.escape(alias)}\s*\(\s*(\d(?:\.\d)?)\s*\)\s*:",
            # Task Achievement: 7.5
            rf"^\s*(?:[-*]\s*)?{re.escape(alias)}\s*:\s*(\d(?:\.\d)?)\s*$",
        ]
        for pat in patterns:
            m = re.search(pat, section_text, flags=re.IGNORECASE | re.MULTILINE)
            if m:
                return clamp_band(m.group(1))
    return np.nan

def extract_score_from_section(section_text):
    patterns = [
        # Suggested Band Score (Task Achievement): 6.5
        r"suggested\s+band\s+score\s*\([^)]*\)\s*:\s*(\d(?:\.\d)?)",
        # Suggested Band Score: 6.5
        r"suggested\s+band\s+score\s*:\s*(\d(?:\.\d)?)",
        # Band Score: 6.5
        r"band\s+score\s*:\s*(\d(?:\.\d)?)",
    ]

    for pat in patterns:
        m = re.search(pat, section_text, flags=re.IGNORECASE)
        if m:
            return clamp_band(m.group(1))

    return np.nan

def extract_sub_scores_only(text):
    text = normalize_eval_text(text)

    result = {
        "TR": np.nan,
        "CC": np.nan,
        "LR": np.nan,
        "GRA": np.nan,
        "n_found": 0,
        "parse_quality": "bad",
    }

    for key in ["TR", "CC", "LR", "GRA"]:
        aliases = CRITERION_ALIASES[key]
        sec = find_section(text, aliases)

        if sec:
            score = extract_score_from_heading(sec, aliases)
            if pd.isna(score):
                score = extract_score_from_section(sec)
            result[key] = score

    n_found = sum(pd.notna(result[k]) for k in ["TR", "CC", "LR", "GRA"])
    result["n_found"] = n_found

    if n_found == 4:
        result["parse_quality"] = "good"
    elif n_found >= 2:
        result["parse_quality"] = "medium"
    elif n_found == 1:
        result["parse_quality"] = "weak"
    else:
        result["parse_quality"] = "bad"

    return pd.Series(result)

def round_ielts_overall(avg_score):
    if pd.isna(avg_score):
        return np.nan
    val = Decimal(str(avg_score)) * Decimal("2")
    return float((val.quantize(Decimal("1"), rounding=ROUND_HALF_UP)) / Decimal("2"))

def compute_overall_from_4(row):
    scores = [row["TR"], row["CC"], row["LR"], row["GRA"]]
    if any(pd.isna(x) for x in scores):
        return np.nan
    avg_score = sum(scores) / 4.0
    return round_ielts_overall(avg_score)

print("--- Đang trích xuất điểm TR, CC, LR, GRA từ evaluation ---")
parsed = df["evaluation"].apply(extract_sub_scores_only)

df[["TR", "CC", "LR", "GRA", "n_found", "parse_quality"]] = parsed

# Chỉ giữ các mẫu parse đủ 4 tiêu chí
df = df[df["n_found"] == 4].reset_index(drop=True)

# Tính overall từ 4 tiêu chí theo rule IELTS
df["overall_raw"] = df[["TR", "CC", "LR", "GRA"]].mean(axis=1)
df["overall"] = df.apply(compute_overall_from_4, axis=1)

print(f"Số lượng sau khi parse đủ 4 tiêu chí: {len(df)}")
print(df[["TR", "CC", "LR", "GRA", "overall"]].head())

--- Đang trích xuất điểm TR, CC, LR, GRA từ evaluation ---
Số lượng sau khi parse đủ 4 tiêu chí: 8016
    TR   CC   LR  GRA  overall
0  5.0  4.5  4.0  4.5      4.5
1  6.5  6.5  6.0  6.0      6.5
2  6.0  5.5  5.0  5.0      5.5
3  3.5  3.0  3.0  3.0      3.0
4  4.5  3.5  3.5  3.5      4.0


In [ ]:
# CELL 6
def get_advanced_features(df):
    # 1. Tính Semantic Relevance giữa Prompt và Essay (S-BERT)
    print("Tính toán Prompt-Essay Relevance...")
    prompt_embeddings = sbert_model.encode(
        df["prompt"].tolist(),
        convert_to_tensor=True,
        show_progress_bar=True
    )
    essay_embeddings = sbert_model.encode(
        df["essay"].tolist(),
        convert_to_tensor=True,
        show_progress_bar=True
    )
    cosine_scores = util.cos_sim(prompt_embeddings, essay_embeddings)
    df["prompt_relevance"] = cosine_scores.diag().cpu().numpy()

    # 2. Tính Lexical Diversity & Readability
    print("Tính toán Linguistic Features...")
    ttr_scores = []
    readability_scores = []

    for doc in tqdm(nlp.pipe(df["essay"], batch_size=50), total=len(df)):
        tokens = [t.text.lower() for t in doc if not t.is_punct and not t.is_space]
        ttr = len(set(tokens)) / len(tokens) if len(tokens) > 0 else 0.0
        ttr_scores.append(ttr)
        readability_scores.append(textstat.flesch_reading_ease(doc.text))

    df["lexical_diversity"] = ttr_scores
    df["readability_score"] = readability_scores
    return df

df = get_advanced_features(df)

Tính toán Prompt-Essay Relevance...


Batches:   0%|          | 0/251 [00:00<?, ?it/s]

Batches:   0%|          | 0/251 [00:00<?, ?it/s]

Tính toán Linguistic Features...


100%|██████████| 8016/8016 [00:31<00:00, 258.01it/s]


In [ ]:
# CELL 8
def create_instruction(row):
    rationale = (
        f"This essay has a lexical diversity of {row['lexical_diversity']:.2f}, "
        f"a readability score of {row['readability_score']:.1f}, "
        f"and its relevance to the prompt is {row['prompt_relevance']:.2f}."
    )

    target_output = (
        f"Analysis: {rationale} "
        f"Detailed Evaluation: {row['evaluation']} "
        f"Scores: TR: {row['TR']}, CC: {row['CC']}, LR: {row['LR']}, GRA: {row['GRA']} "
        f"Final Band: {row['overall']}"
    )
    return target_output

df["target_text"] = df.apply(create_instruction, axis=1)

if "band" in df.columns:
    df = df.drop(columns=["band"])

df = df.rename(columns={"overall": "band"})

In [ ]:
# CELL 10
def safe_stratify_labels(series, min_count=2):
    counts = series.value_counts(dropna=False)
    if len(counts) <= 1:
        return None
    if counts.min() < min_count:
        return None
    return series

stratify_labels = safe_stratify_labels(df["overall"])

train_df, temp_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=stratify_labels,
)

temp_stratify = safe_stratify_labels(temp_df["overall"])

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.5,
    random_state=42,
    stratify=temp_stratify,
)

print(f"\nKích thước tập Train: {len(train_df)}")
print(f"Kích thước tập Val: {len(val_df)}")
print(f"Kích thước tập Test: {len(test_df)}")

train_df.to_csv("ielts_train_df2.csv", index=False)
val_df.to_csv("ielts_val_df2.csv", index=False)
test_df.to_csv("ielts_test_df2.csv", index=False)


Kích thước tập Train: 6412
Kích thước tập Val: 802
Kích thước tập Test: 802
